# NB-06: T-5 数値位置取り予測（正規化着順）

**ターゲット**: `finish_position / field_size`（0=1着, 1=最下位）の正規化着順を回帰予測する。  
**モデル**: LightGBM Regression  
**主評価指標**: Spearman 相関, MAE

### 活用方法
- 騎手・馬の「コース適性」や「距離適性」を数値化した中間予測
- T-1 の入力特徴量として使用（位置取りが良い馬ほど勝率が高い）


In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/work/keiba-vpn")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.pipeline.models.notebook_utils import (
    load_master, encode_cats, feature_set,
    train_lgb, oof_predict, eval_regression, save_oof,
    DEFAULT_PARAMS_REGRESSION, SURFACE_CATS,
)
print("imports OK")


## 1. データ読み込みとターゲット生成

In [ ]:
df = load_master()

# 正規化着順: finish_position / field_size (0=1着, 1=最下位)
df["fp_norm"] = pd.to_numeric(df["finish_position"], errors="coerce") / pd.to_numeric(df["field_size"], errors="coerce")
df_clean = df[df["fp_norm"].between(0, 1)].copy()

print(f"fp_norm 範囲: {df_clean['fp_norm'].min():.3f} ~ {df_clean['fp_norm'].max():.3f}")
print(f"fp_norm 平均: {df_clean['fp_norm'].mean():.3f} (0.5 に近いほど中央付近)")
print(f"データ数: {len(df_clean):,}")
df["surface_cat"] = df["surface_cat"].astype(str)  # Categorical → str


## 2. 特徴量定義・モデル学習

In [ ]:
FEAT_T5 = feature_set(df_clean, extra=[
    "speed_max", "speed_avg", "speed_distance", "speed_recent_1",
])
CAT_USE = [c for c in ["venue","surface","direction","grade","track_condition","weather","sex"] if c in df_clean.columns]
df_clean = encode_cats(df_clean, CAT_USE)

params_t5 = {**DEFAULT_PARAMS_REGRESSION}
models_t5: dict = {}
oof_all_t5 = pd.DataFrame()

for sc in SURFACE_CATS:
    df_sc = df_clean[df_clean["surface_cat"] == sc].copy()
    df_tr = df_sc[df_sc["split"] == "train"]
    df_vl = df_sc[df_sc["split"] == "valid"]
    print(f"\n=== {sc} ===  train={len(df_tr):,}  valid={len(df_vl):,}")
    if len(df_tr) < 100: print("  スキップ"); continue

    model = train_lgb(df_tr, df_vl, FEAT_T5, "fp_norm", params_t5, cat_features=CAT_USE)
    models_t5[sc] = model

    pred_vl = model.predict(df_vl[FEAT_T5])
    metrics = eval_regression(df_vl["fp_norm"].values, pred_vl)
    sc_r, _ = spearmanr(df_vl["fp_norm"].dropna(), pred_vl[:df_vl["fp_norm"].notna().sum()])
    print(f"  Valid: MAE={metrics['mae']:.4f}  Spearman={sc_r:.4f}")

    oof = oof_predict(df_tr, FEAT_T5, "fp_norm", params_t5, cat_features=CAT_USE)
    oof_df = df_tr[["race_id","horse_number","surface_cat","fp_norm","split"]].copy()
    oof_df["t5_oof"] = oof.values
    oof_all_t5 = pd.concat([oof_all_t5, oof_df], ignore_index=True)

print("\nモデル学習完了:", list(models_t5.keys()))


## 3. テスト評価 & OOF 保存

In [ ]:
for sc, model in models_t5.items():
    df_te = df_clean[(df_clean["surface_cat"] == sc) & (df_clean["split"] == "test")]
    if df_te.empty: continue
    pred_te = model.predict(df_te[FEAT_T5])
    metrics = eval_regression(df_te["fp_norm"].values, pred_te)
    sc_r, _ = spearmanr(df_te["fp_norm"].dropna(), pred_te[:df_te["fp_norm"].notna().sum()])
    print(f"[{sc}] Test: MAE={metrics['mae']:.4f}  RMSE={metrics['rmse']:.4f}  Spearman={sc_r:.4f}")

if not oof_all_t5.empty:
    oof_all_t5.to_parquet("/home/jovyan/work/keiba-vpn/data/local/modeling/oof/t5_oof.parquet", index=False)
    print(f"T-5 OOF saved: shape={oof_all_t5.shape}")

best_sc = max(models_t5, key=lambda s: models_t5[s].num_trees()) if models_t5 else None
if best_sc:
    imp = pd.Series(models_t5[best_sc].feature_importance(importance_type="gain"), index=FEAT_T5)
    imp.sort_values(ascending=False).head(20).plot.barh(figsize=(8,6),
        title=f"T-5 Feature Importance [{best_sc}]")
    plt.tight_layout(); plt.show()
